## Citations
* Python code from past assignments in the courses that I took during my past years in college: INFO 2000 and INFO 3000 and CSCI 4900, 
  * INFO 2000: https://github.com/thespcrewroy/Public-College-Portfolio/tree/feb06786a2b200e7f1269a43cb536eb49fa7f8d5/INFO%202000
  * INFO 3000: https://github.com/thespcrewroy/Public-College-Portfolio/tree/feb06786a2b200e7f1269a43cb536eb49fa7f8d5/INFO%203000
  * CSCI 4900: https://github.com/thespcrewroy/Public-College-Portfolio/tree/feb06786a2b200e7f1269a43cb536eb49fa7f8d5/CSCI%204900
* ChatGPT5 (in-line citations are stated below. use CTRL+F "ChatGPT")
* Professor Mottalib ANN Demo

## Part 1

* Generate an intelligent irrigation system model
* Measures the moisture of soil
* Make a decision to turn the water supply on or off
* Dataset: “pump-data.csv”.
    * 4 columns: Crop, Moisture, Temp, Pump.
    * 200 records
* Classification Task: predict the label Pump (1/0)

* Read the data and remove any record with missing values (if any)
* Randomly split the dataset into train-test with a ratio of 60:40.
* Implement a K-Nearest Neighbor classifier and train the model with the
training data
* Report the following metrics
  * K = 3
    * Accuracy
    * Precision
    * Recall
    * F1-score
  * K = 5
    * Accuracy
    * Precision
    * Recall
    * F1-score
  * K = 7
    * Accuracy
    * Precision
    * Recall
    * F1-score
  * Distance Measures
    * Manhattan distance
    * Euclidean distance

In [1]:
'''Import Libraries'''
import os # for operating system dependent functionality
import math # for mathematical functions
import numpy as np # for numerical operations
import pandas as pd # for data manipulation
import matplotlib as plt # for plotting
import seaborn as sns # for data visualization
from collections import Counter # for counting hashable objects

In [2]:
'''Read the CSV and look at the shape'''
df = pd.read_csv('pump_data.csv') # load the dataset
print("First 10 rows:\n", df.head(10), "\n") # display the first 10 rows
print(f"Shape before dropping missing values: {df.shape}\n") # display the shape of the dataframe (200 records, 4 columns)

First 10 rows:
      crop  moisture  temp  pump
0  cotton       638    16     1
1  cotton       522    18     1
2  cotton       741    22     1
3  cotton       798    32     1
4  cotton       690    28     1
5  cotton       558    23     1
6  cotton       578    12     1
7  cotton       673    35     1
8  cotton       642    45     1
9  cotton       723    11     1 

Shape before dropping missing values: (200, 4)



In [3]:
'''Determine Missing Values'''
missing_values = df.isnull().sum() # count the number of missing values in each column
print(f"Missing Values:\n {missing_values}") # print the count of missing values

Missing Values:
 crop        0
moisture    0
temp        0
pump        0
dtype: int64


In [4]:
'''Remove Missing Values'''
df = df.dropna() # remove rows with missing values
print(f"Shape after dropping missing values: {df.shape}\n") # display the shape of the dataframe after dropping missing values
print(df.isnull().sum()) # verify that there are no missing values left

Shape after dropping missing values: (200, 4)

crop        0
moisture    0
temp        0
pump        0
dtype: int64


In [5]:
'''Train-Test Split'''
np.random.seed(42) # for reproducibility
shuffled_indices = np.random.permutation(len(df)) # chatgpt5 (2025, September, "how to shuffle the indices of this dataset?"").
train_size = int(0.6 * len(df)) # 60% for training
train_indices = shuffled_indices[:train_size] # first 60% for training
test_indices = shuffled_indices[train_size:] # remaining 40% for testing

# Split the data
train_df = df.iloc[train_indices] # training set
test_df = df.iloc[test_indices] # testing set

# Separate features and labels for training and testing
X_train = train_df.drop('pump', axis=1) # features for training
y_train = train_df['pump'] # labels for training
X_test = test_df.drop('pump', axis=1) # features for testing
y_test = test_df['pump'] # labels for testing

# Print the different sets
print(f"Training Set Shape: {train_df.shape}") # shape of training set
print("\nTraining Features:\n", X_train.head())
print(f"\nTesting Set Shape: {test_df.shape}") # shape of testing set
print("\nTesting Features:\n", X_test.head())

Training Set Shape: (120, 4)

Training Features:
        crop  moisture  temp
95   cotton       843    41
15   cotton       716    25
30   cotton       661    35
158  cotton       865    17
128  cotton       462    32

Testing Set Shape: (80, 4)

Testing Features:
        crop  moisture  temp
40   cotton       698    45
108  cotton       482    27
155  cotton      1020    27
156  cotton       933    42
25   cotton       833    13


In [6]:
'''ChatGPT4 - (2025, September, "Help me implement KNN based off of Mottalib Slides and his perceptron code. ")'''
class KNN:
    def __init__(self, k=3, metric='euclidean'):
        self.k = k # number of neighbors
        self.metric = metric # distance metric
        self.X = None # training data
        self.Y = None # training labels

    # distance helper (like your activation helper)
    def _distance(self, A, b):
        """
        A: (n, d) training matrix
        b: (d,)   one test row
        returns distances from b to each row in A
        """
        if self.metric == 'manhattan':
            return np.sum(np.abs(A - b), axis=1)
        # default: euclidean
        diff = A - b
        return np.sqrt(np.sum(diff * diff, axis=1))

    def fit(self, X, Y):
        """
        'Training' for KNN = just remember the data.
        """
        self.X = np.asarray(X)
        self.Y = np.asarray(Y)

    def predict(self, X):
        """
        For each test point:
         1) compute distances to all train points
         2) pick indices of k smallest distances
         3) majority vote on those labels (ties broken by smallest label)
        """
        X = np.asarray(X)
        preds = []
        for i in range(X.shape[0]):
            d = self._distance(self.X, X[i])
            nn = np.argpartition(d, self.k)[:self.k]
            neigh_labels = self.Y[nn]

            # majority vote; works for 0/1 or multi-class
            values, counts = np.unique(neigh_labels, return_counts=True)
            preds.append(values[np.argmax(counts)])
        return np.array(preds)

    # optional, mirrors your perceptron usage pattern
    def score(self, X, Y):
        pred = self.predict(X)
        return np.mean(pred == np.asarray(Y))


In [7]:
'''ChatGPT - (2025, September, "Help me OneHot Encode the categorical features in this dataset so as to use be able to use them in KNN")'''
y_train = y_train.astype(int) # ensure labels are integers
y_test  = y_test.astype(int) # ensure labels are integers
cat_cols = X_train.select_dtypes(include=['object']).columns # identify categorical columns
X_train_enc = pd.get_dummies(X_train, columns=cat_cols, drop_first=False) # one-hot encode categorical features
X_test_enc  = pd.get_dummies(X_test,  columns=cat_cols, drop_first=False) # one-hot encode categorical features
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0) # align test set columns with training set
print("Encoded feature columns:", list(X_train_enc.columns)) # print the columns after encoding

Encoded feature columns: ['moisture', 'temp', 'crop_cotton']


In [ ]:

'''Calculate Metrics'''

def _safe_div(a, b): # safe division to avoid division by zero
    return a / b if b != 0 else 0.0 # return 0.0 if denominator is zero

def binary_metrics(y_true, y_pred, positive=1): # positive class label
    y_true = np.asarray(y_true) # ensure numpy arrays
    y_pred = np.asarray(y_pred) # ensure numpy arrays
    tp = np.sum((y_true == positive) & (y_pred == positive)) # true positives
    tn = np.sum((y_true != positive) & (y_pred != positive)) # true negatives
    fp = np.sum((y_true != positive) & (y_pred == positive)) # false positives
    fn = np.sum((y_true == positive) & (y_pred != positive)) # false negatives

    precision = _safe_div(tp, tp + fp) # precision
    recall    = _safe_div(tp, tp + fn) # recall
    f1        = _safe_div(2 * precision * recall, precision + recall) # F1 score
    accuracy  = _safe_div(tp + tn, tp + tn + fp + fn) # accuracy

    return accuracy, precision, recall, f1 # return all metrics


# Run KNN with different parameters and collect results
X_train_enc = X_train_enc.astype(float) # ensure features are float
X_test_enc  = X_test_enc.astype(float) # ensure features are float

results = [] # to store results
for metric in ["euclidean", "manhattan"]: # different distance metrics
    for k in [3, 5, 7]: # different k values
        knn = KNN(k=k, metric=metric) # initialize KNN
        knn.fit(X_train_enc.values, y_train.values) # <-- encoded
        y_pred = knn.predict(X_test_enc.values) # <-- encoded

        acc, prec, rec, f1 = binary_metrics(y_test.values, y_pred, positive=1) # compute metrics
        results.append([metric, k, round(acc,4), round(prec,4), round(rec,4), round(f1,4)]) # store results

report = pd.DataFrame(results, columns=["Distance", "k", "Accuracy", "Precision", "Recall", "F1"]) # create results dataframe of results
print(report) # print the results

    Distance  k  Accuracy  Precision  Recall      F1
0  euclidean  3    0.9875     0.9844     1.0  0.9921
1  euclidean  5    0.9750     0.9692     1.0  0.9844
2  euclidean  7    0.9750     0.9692     1.0  0.9844
3  manhattan  3    0.9875     0.9844     1.0  0.9921
4  manhattan  5    0.9750     0.9692     1.0  0.9844
5  manhattan  7    0.9750     0.9692     1.0  0.9844


## Part 2

* Dataset: “Invistico_Airline.csv”
  * 22 columns in total
  * 130k records
  * Class Label: satisfaction (satisfied/dissatisfied)
* Tasks
  * Read the data and keep only these columns: [“Customer Type”, “Type of Travel”, “Class”]
  * Split the dataset into train-test with a ratio of 75:25.
  * Build a decision tree classifier with these specific design choices:
    * Splitting criterion: Entropy
    * Stopping condition: either pure node or no attribute left
      * If no attribute left, assign majority class.
* Report the results with discussion.

In [ ]:
'''Load the Dataset for Decision Tree and Entropy/Information Gain'''
df = pd.read_csv("Invistico_Airline.csv") # load the dataset
df.columns = [c.strip() for c in df.columns] # clean column names

FEATURES = ["Customer Type", "Type of Travel", "Class"] # categorical features
TARGET   = "satisfaction" # target variable
df = df[FEATURES + [TARGET]].dropna().copy() # subset and drop missing values
X = df[FEATURES] # features
y = df[TARGET] # labels

In [10]:
'''Train/Test Split (75:25)'''
rng = np.random.default_rng(42) # seeded random generator for reproducibility
idx = np.arange(len(df)) # indices of the dataset
rng.shuffle(idx) # shuffle indices
cut = int(0.75 * len(idx)) # 75% for training
train_idx, test_idx = idx[:cut], idx[cut:] # split indices

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx] # training set
X_test,  y_test  = X.iloc[test_idx],  y.iloc[test_idx] # testing set

In [ ]:
'''Entropy and Information Gain - ChatGPT - (2025, September, "Help me implement Entropy and Information Gain for Decision Trees")'''
def entropy(labels): # H(S) = - sum_c (|S_c|/|S|) * log2(|S_c|/|S|)
    counts = Counter(labels) # count occurrences of each label
    n = len(labels) # total number of labels
    if n == 0: # avoid log(0)
        return 0.0 # return 0.0
    return -sum((c/n) * np.log2(c/n) for c in counts.values() if c > 0) # entropy formula

def info_gain_categorical(parent_y, partitions): # Information Gain for Categorical Features
    H_parent = entropy(parent_y) # entropy of parent
    n = len(parent_y) # total number of samples
    weighted_child_entropy = 0.0 # initialize weighted child entropy
    for part in partitions: # for each partition
        nv = len(part) # number of samples in partition
        if nv == 0: # avoid log(0)
            continue # skip empty partitions
        weighted_child_entropy += (nv / n) * entropy(part) # weighted entropy
    return H_parent - weighted_child_entropy # Information Gain formula

In [12]:
'''Decision Tree Implementation - ChatGPT - (2025, September, "Help me implement a Decision Tree from scratch using Entropy and Information Gain")'''
class DecisionNode: # tree node
    def __init__(self, feature=None, branches=None, prediction=None, majority=None): # initialize node
        self.feature    = feature          # feature name this node splits on (None if leaf)
        self.branches   = branches or {}   # dict: category value -> subtree
        self.prediction = prediction       # label if leaf, else None
        self.majority   = majority         # majority label at this node (for fallbacks)

def majority_label(labels): # majority label in a list
    return Counter(labels).most_common(1)[0][0] # return the most common label

def build_tree(X, y): # recursive tree builder
    unique_labels = set(y) # unique labels in current node
    if len(unique_labels) == 1: # stop if pure
        lbl = next(iter(unique_labels)) # only label
        return DecisionNode(prediction=lbl, majority=lbl) # leaf node

    if X.shape[1] == 0: # no features to split on
        maj = majority_label(y) # majority label
        return DecisionNode(prediction=maj, majority=maj) # leaf node

    best_feat, best_gain = None, -1.0 # initialize best feature and gain
    for feat in X.columns: # for each feature
        parts = [] # partitions for this feature
        for val in X[feat].unique(): # for each unique value of the feature
            parts.append(y[X[feat] == val]) # partition labels
        gain = info_gain_categorical(y, parts) # compute information gain
        if gain > best_gain: # if better gain found
            best_gain, best_feat = gain, feat # update best gain and feature

    if best_gain <= 1e-12: # no informative split
        maj = majority_label(y) # majority label
        return DecisionNode(prediction=maj, majority=maj) # leaf node

    node = DecisionNode(feature=best_feat, majority=majority_label(y)) # create decision node
    for val in X[best_feat].unique(): # for each unique value of the best feature
        mask = (X[best_feat] == val) # mask for this branch
        X_sub = X.loc[mask].drop(columns=[best_feat]) # subset features without the best feature
        y_sub = y.loc[mask] # subset labels
        node.branches[val] = build_tree(X_sub, y_sub) # recursive build subtree
    return node # return the node

def predict_one(node, row_dict): # predict for one instance
    while node.prediction is None: # while not a leaf
        val = row_dict.get(node.feature, None) # get feature value
        if val in node.branches: # if branch exists
            node = node.branches[val] # go to subtree
        else: # unseen category at test time
            return node.majority # fallback to majority
    return node.prediction # return prediction

def predict(tree, X): # predict for multiple instances
    records = X.to_dict(orient="records") # convert to list of dicts
    return [predict_one(tree, r) for r in records] # predict for each record

In [13]:
'''Decision Tree Training, Prediction, and Evaluation'''
tree = build_tree(X_train, y_train) # train the decision tree
y_pred = predict(tree, X_test) # predict on test set

def classification_report(y_true, y_pred): # detailed classification report
    y_true = np.array(y_true) # ensure numpy arrays
    y_pred = np.array(y_pred) # ensure numpy arrays
    labels = sorted(set(y_true)) # unique labels
    acc = float(np.mean(y_true == y_pred)) # overall accuracy

    per_class = {} # per-class metrics
    for lbl in labels: # for each label
        TP = int(np.sum((y_true == lbl) & (y_pred == lbl))) # true positives
        FP = int(np.sum((y_true != lbl) & (y_pred == lbl))) # false positives
        FN = int(np.sum((y_true == lbl) & (y_pred != lbl))) # false negatives
        prec = TP / (TP + FP) if (TP + FP) else 0.0 # precision
        rec  = TP / (TP + FN) if (TP + FN) else 0.0 # recall
        f1   = (2 * prec * rec) / (prec + rec) if (prec + rec) else 0.0 # F1 score
        per_class[lbl] = {"precision": prec, "recall": rec, "f1": f1, "support": int(np.sum(y_true == lbl))} # store metrics
    return acc, per_class # return accuracy and per-class metrics

acc, report = classification_report(y_test, y_pred) # generate classification report

print(f"Decision Tree (Entropy) — test accuracy: {acc*100:.2f}%\n") # print accuracy
for lbl, m in report.items(): # print per-class metrics
    print(f"{lbl:>13}: precision={m['precision']:.3f}  recall={m['recall']:.3f}  f1={m['f1']:.3f}  support={m['support']}") # print metrics


Decision Tree (Entropy) — test accuracy: 66.77%

 dissatisfied: precision=0.597  recall=0.812  f1=0.688  support=14655
    satisfied: precision=0.780  recall=0.549  f1=0.644  support=17815


## Part 3

* Train Dataset: “perceptron-train.csv” (299 records)
* Test Dataset: “perceptron-test.csv” (199 record)
* Datasets
  * Feature Columns: x1, x2
  * Label Column: Output
* Tasks
  * Remove missing values
  * Use Perceptron learning algorithm to train a Perceptron with the following design choices:
    * Learning rate: [ 0.005, 0.01, 0.05 ]
    * Weight initialization: random with specific seed value for reproducibility
    * Activation function: [ Step, Sign, Sigmoid ]
  * Report the results with discussion. Explain the effects of each change in parameter.

In [14]:
'''Source: Professor Mottalib's ANN Perceptron Code with Enhancements for Activation Functions and Seeding'''
train_df = pd.read_csv("perceptron-train.csv").dropna().reset_index(drop=True) # load and clean training data
test_df  = pd.read_csv("perceptron-test.csv").dropna().reset_index(drop=True) # load and clean testing data

X_train = train_df[["x1","x2"]].to_numpy(dtype=float) # convert features to numpy array
y_train = train_df["Output"].to_numpy().astype(int).ravel() # convert labels to numpy array
X_test  = test_df[["x1","x2"]].to_numpy(dtype=float) # convert features to numpy array
y_test  = test_df["Output"].to_numpy().astype(int).ravel() # convert labels to numpy array

class Perceptron:
    def __init__( self, learning_rate = 0.05, epochs = 1000, activation="sign", seed=42): # added activation and seed
        self.lr = learning_rate # learning rate
        self.epochs = epochs # number of epochs
        self.weights = None # weights will be initialized during training
        self.bias = None # bias will be initialized during training
        self.activation_fn = activation.lower() # activation function
        self.seed = seed # random seed for reproducibility

    def _sigmoid(self, z): # helper for sigmoid activation
        return 1.0 / (1.0 + np.exp(-np.clip(z, -50, 50))) # clip to avoid overflow

    def activation(self, z, proba=False):
        if self.activation_fn == "step" or self.activation_fn == "sign":
            return 1 if z >= 0 else -1
        if self.activation_fn == "sigmoid":
            p = self._sigmoid(z)
            return p if proba else (1 if p >= 0.5 else -1)
        raise ValueError(f"Unknown activation: {self.activation_fn}")

    def fit( self, X, Y ): # training function
        rng = np.random.default_rng(self.seed) # for reproducibility
        self.weights = rng.normal(0, 0.01, X.shape[1]) # small random weights
        self.bias = float(rng.normal(0, 0.01)) # small random bias

        for epoch in range( self.epochs ): # for each epoch
            for i in range( X.shape[0] ): # for each sample
                y_pred = self.activation( np.dot( self.weights, X[i] ) + self.bias ) # predict
                error = Y[i] - y_pred # compute error
                self.weights = self.weights + self.lr * error * X[i] # update weights
                self.bias = self.bias + self.lr * error # update bias

    def predict( self, X ): # prediction function
        y_pred = [] # list to store predictions
        for i in range( X.shape[0] ):   # for each sample
            y_pred.append( self.activation( np.dot( self.weights, X[i] ) + self.bias )) # predict
        return np.array( y_pred ) # return predictions as numpy array

def calculate_accuracy( y_true, y_pred ): # accuracy calculation
    if len(y_true) != len(y_pred): # check lengths
        raise ValueError("Lengths of y_true and y_pred must be equal.") # raise error if not equal

    correct_predictions = 0 # counter for correct predictions
    for true_label, predicted_label in zip( y_true, y_pred ): # for each true and predicted label
        if true_label == predicted_label: # if they match
            correct_predictions += 1 # increment counter if correct

    accuracy = correct_predictions / len( y_true ) # compute accuracy
    return accuracy # return accuracy

# Run experiments with different hyperparameters
lr = [0.005, 0.01, 0.05]  # learning rates to try
activations = ["step", "sign", "sigmoid"]  # activation functions to try
epochs = 1000 # fixed number of epochs
seed = 42 # fixed random seed

for act in activations:
    for l in lr: # for each learning rate
        clf = Perceptron(learning_rate=l, epochs=epochs, activation=act, seed=seed) # create perceptron with given params
        clf.fit(X_train, y_train) # fit on training data
        Y_predict = clf.predict(X_test) # predict on test data
        print(f'Activation: {act}, Learning rate: {l}, Epochs: {epochs}, Seed: {seed}') # print hyperparameters
        print(f'Accuracy: {calculate_accuracy(y_test, Y_predict) * 100:.2f}%') # print accuracy
        print("=================") # separator

Activation: step, Learning rate: 0.005, Epochs: 1000, Seed: 42
Accuracy: 71.50%
Activation: step, Learning rate: 0.01, Epochs: 1000, Seed: 42
Accuracy: 67.50%
Activation: step, Learning rate: 0.05, Epochs: 1000, Seed: 42
Accuracy: 71.50%
Activation: sign, Learning rate: 0.005, Epochs: 1000, Seed: 42
Accuracy: 71.50%
Activation: sign, Learning rate: 0.01, Epochs: 1000, Seed: 42
Accuracy: 67.50%
Activation: sign, Learning rate: 0.05, Epochs: 1000, Seed: 42
Accuracy: 71.50%
Activation: sigmoid, Learning rate: 0.005, Epochs: 1000, Seed: 42
Accuracy: 71.50%
Activation: sigmoid, Learning rate: 0.01, Epochs: 1000, Seed: 42
Accuracy: 67.50%
Activation: sigmoid, Learning rate: 0.05, Epochs: 1000, Seed: 42
Accuracy: 71.50%
